In [1]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
from src.utils import month_to_season
START, END = "2020-01-01", "2025-12-31"
pollution = pd.read_csv("../data/raw/air_poll_ITO_(2018-2025).csv")
weather = pd.read_csv("../data/raw/weather_dataset_delhi_2020_2026.csv")
pollution.columns = pollution.columns.str.strip()
weather.columns = weather.columns.str.strip()
print(pollution.shape, weather.shape)
print(pollution.columns.tolist())
print(weather.columns.tolist())
# required columns must exist (adjust names if your file differs)
req_w = ["time", "temperature", "humidity", "dew_point", "feels_like", "rain",
"pressure", "cloud_cover", "wind_speed", "wind_direction"]
req_p = ["Day", "PM2.5", "PM10", "NO2", "SO2"]
assert not set(req_w)- set(weather.columns), set(req_w)- set(weather.columns)
assert not set(req_p)- set(pollution.columns), set(req_p)- set(pollution.columns)

(2923, 5) (57768, 20)
['Day', 'PM2.5', 'PM10', 'NO2', 'SO2']
['time', 'temperature', 'humidity', 'dew_point', 'feels_like', 'rain', 'pressure', 'cloud_cover', 'wind_speed', 'wind_direction', 'weather_code', 'year', 'month', 'day', 'hour', 'day_name', 'is_weekend', 'temperature_difference', 'rainfall_flag', 'season']


In [4]:
pollution["Date"] = pd.to_datetime(pollution["Day"], dayfirst=True, errors="coerce")
bad = pollution[pollution["Date"].isna()]
print("Invalid dates:", len(bad))
# inspect these rows BEFORE continuing
pollution = pollution.dropna(subset=["Date"])
# force numeric (strings such as "NA" or "-" become NaN)
for c in ["PM2.5", "PM10", "NO2", "SO2"]:
    pollution[c] = pd.to_numeric(pollution[c], errors="coerce")
pollution = pollution[(pollution["Date"] >= START) & (pollution["Date"] <= END)]
pollution = pollution[["Date", "PM2.5", "PM10", "NO2", "SO2"]]
dups = pollution["Date"].duplicated().sum()
print("Duplicate pollution dates:", dups)
assert dups == 0, "Resolve duplicate dates (inspect, then aggregate deliberately)."

Invalid dates: 0
Duplicate pollution dates: 0


In [6]:
weather["time"] = pd.to_datetime(weather["time"])
weather = weather[(weather["time"] >= f"{START} 00:00") &
(weather["time"] <= f"{END} 23:00")].copy()
print("Duplicate timestamps:", weather["time"].duplicated().sum())
weather = weather.drop_duplicates(subset="time").sort_values("time")
full_hours = pd.date_range(f"{START} 00:00", f"{END} 23:00", freq="h")
missing_hours = full_hours.difference(weather["time"])
print("Expected hours:", len(full_hours))
# 52608
print("Observed hours:", len(weather))
print("Missing hours :", len(missing_hours))
for c in req_w[1:]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")
weather["Date"] = weather["time"].dt.normalize()

Duplicate timestamps: 0
Expected hours: 52608
Observed hours: 52608
Missing hours : 0


In [7]:
theta = np.deg2rad(weather["wind_direction"])
weather["wind_u"] =-weather["wind_speed"] * np.sin(theta)
weather["wind_v"] =-weather["wind_speed"] * np.cos(theta)
weather_daily =weather.groupby("Date").agg(
weather_hours=("time", "size"),
temperature_mean=("temperature", "mean"),
temperature_min=("temperature", "min"),
temperature_max=("temperature", "max"),
humidity_mean=("humidity", "mean"),
dew_point_mean=("dew_point", "mean"),
feels_like_mean=("feels_like", "mean"),
rain_sum=("rain","sum"),
pressure_mean=("pressure", "mean"),
cloud_cover_mean=("cloud_cover", "mean"),
wind_speed_mean=("wind_speed", "mean"),
wind_speed_max=("wind_speed", "max"),
wind_u_mean=("wind_u", "mean"),
wind_v_mean=("wind_v", "mean"),
).reset_index()
weather_daily["wind_speed_vector"] = np.hypot(
weather_daily["wind_u_mean"], weather_daily["wind_v_mean"])
weather_daily["wind_direction_vector"] = np.degrees(np.arctan2(-weather_daily["wind_u_mean"],-weather_daily["wind_v_mean"])) % 360
print(weather_daily["weather_hours"].value_counts().sort_index())

weather_hours
24    2192
Name: count, dtype: int64


In [8]:
calendar = pd.DataFrame({"Date": pd.date_range(START, END, freq="D")})
df = (calendar.merge(pollution, on="Date", how="left").merge(weather_daily, on="Date", how="left").sort_values("Date").reset_index(drop=True))
assert len(df)== 2192
assert df["Date"].diff().dropna().eq(pd.Timedelta(days=1)).all()
print(df.shape,df["Date"].min(), df["Date"].max())
print(df.isna().sum())

(2192, 21) 2020-01-01 00:00:00 2025-12-31 00:00:00
Date                       0
PM2.5                     62
PM10                     207
NO2                       53
SO2                       28
weather_hours              0
temperature_mean           0
temperature_min            0
temperature_max            0
humidity_mean              0
dew_point_mean             0
feels_like_mean            0
rain_sum                   0
pressure_mean              0
cloud_cover_mean           0
wind_speed_mean            0
wind_speed_max             0
wind_u_mean                0
wind_v_mean                0
wind_speed_vector          0
wind_direction_vector      0
dtype: int64


In [9]:
df["year"] = df["Date"].dt.year
df["month"] =df["Date"].dt.month
df["day_of_week"]= df["Date"].dt.dayofweek
df["quarter"] =df["Date"].dt.quarter
df["is_weekend"]= (df["day_of_week"] >=5).astype(int)
df["season"] =df["month"].map(month_to_season)

In [10]:
out = "../data/processed/delhi_pollution_weather_daily_2020_2025.csv"
df.to_csv(out, index=False)
print("Saved:", out)

Saved: ../data/processed/delhi_pollution_weather_daily_2020_2025.csv
